# 03 — Multi-Tracker Evaluation: ByteTrack vs BotSort

Ultralytics ships two built-in trackers:
- **ByteTrack** — fast, IoU-based, reliable baseline
- **BotSort** — adds Re-ID + Kalman; often better ID consistency

This notebook:
1. Uses the **final** YOLO11 model from `02_train_yolo.ipynb` (all 80% data)
2. Runs a hyperparameter search for each tracker on the **test set**
3. Evaluates MOTA / IDF1 / ID-Switch per tracker using `motmetrics`
4. Produces static visualizations (annotated frames, trajectory plots, metric bar charts)
5. Provides an **interactive real-time tracking** cell (press `q` to quit)

Prerequisites: `01_prepare_dataset.ipynb` + `02_train_yolo.ipynb`

## 0 — Configuration

In [1]:
from pathlib import Path
import json, tempfile, itertools, random
import cv2
import numpy as np

REPO_ROOT    = Path("../")
MOT_DIR      = REPO_ROOT / "data/mot_dataset"
MOT_LV_DIR   = REPO_ROOT / "data/mot_dataset_longvideo"
YOLO_RUNS    = (REPO_ROOT / "runs/yolo11").resolve()
TRACKER_RUNS = (REPO_ROOT / "runs/trackers").resolve()
TRACKER_RUNS.mkdir(parents=True, exist_ok=True)

CHUNKS_DIR = REPO_ROOT / "data/processed/labelstudiochunks"

# Detection thresholds
CONF = 0.20
IOU  = 0.7

# Trackers to compare — both ship with Ultralytics, no extra install needed
TRACKERS = ["bytetrack.yaml", "botsort.yaml"]

# Fraction of test videos to use during HP search (1.0 = all, 0.3 = 30% fastest)
# Final evaluation (Section 3) always uses 100% of test videos.
HP_SEARCH_SUBSET = 0.4

# Hyperparameter grid searched on the test set
HP_GRID = {
    # ── Detection thresholds ──────────────────────────────────────────────
    "det_conf":          [0.15, 0.20, 0.30],
    "det_iou":           [0.5,  0.7],
    # ── Tracker params ────────────────────────────────────────────────────
    "track_high_thresh": [0.25, 0.40],
    "track_low_thresh":  [0.10, 0.20],
    "new_track_thresh":  [0.30, 0.45],
    "track_buffer":      [20,   40],
    "match_thresh":      [0.7,  0.85],
    # ── Post-processing: cross-class NMS IoU threshold (None = disabled) ──
    "pp_cc_iou":         [None, 0.4, 0.6],
    # ── Post-processing: merge time gap in frames (None = disabled) ───────
    "pp_merge_time":     [None, 20, 40],
    # ── Post-processing: merge spatial distance % (only when merge on) ────
    "pp_merge_dist":     [20.0, 40.0],
    # ── Post-processing: interpolate gap frames after merge ───────────────
    "pp_merge_interp":   [True, False],
}

# Max HP combinations to try per tracker (random subsample if grid is large)
HP_MAX_TRIALS = 8

# Metric used to select the best HP configuration.
# Options: "MOTA"  (maximize — classic MOT quality)
#          "IDF1"  (maximize — identity consistency)
#          "ID_Sw" (minimize — fewer ID switches)
#          "ACE"   (minimize — mean absolute count error across classes)
HP_SEARCH_METRIC = "MOTA"   # ← change to "ACE" to optimise for count accuracy

with open(REPO_ROOT / "data/splits/splits.json") as f:
    splits = json.load(f)
CLASSES = splits["classes"]

# Final model weights produced by 02_train_yolo.ipynb Section 2
FINAL_WEIGHTS = YOLO_RUNS / "final" / "weights" / "best.pt"

print("Config OK")
print(f"  Trackers       : {TRACKERS}")
print(f"  Final weights  : {FINAL_WEIGHTS}")
print(f"  HP_MAX_TRIALS  : {HP_MAX_TRIALS}")
print(f"  Classes        : {CLASSES}")


Config OK
  Trackers       : ['bytetrack.yaml', 'botsort.yaml']
  Final weights  : /Users/joh11678/.ws/KARTector/runs/yolo11/final/weights/best.pt
  HP_MAX_TRIALS  : 8
  Classes        : ['Boost', 'Charge', 'Defense', 'Glide', 'HP', 'Offense', 'Top Speed', 'Turn', 'Weight']


## 1 — Helpers

In [ ]:
from ultralytics import YOLO
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import yaml as _yaml

# ── colour palette ────────────────────────────────────────────────────────
_PALETTE = [
    (255, 56,  56),  (255, 157, 151), (255, 112,  31), (255, 178, 29),
    (207, 210,  49), (72,  249, 10),  (146, 204, 23),  (61,  219, 134),
    (26,  147, 52),  (0,   212, 187),
]
def class_color(cid):
    return _PALETTE[int(cid) % len(_PALETTE)]

def draw_tracks(frame, boxes, track_ids, class_ids, class_names, confidences=None):
    out = frame.copy()
    for i, (box, tid, cid) in enumerate(zip(boxes, track_ids, class_ids)):
        x1, y1, x2, y2 = map(int, box)
        color = class_color(cid)[::-1]   # RGB -> BGR
        cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
        label = f"{class_names[int(cid)]} #{int(tid)}"
        if confidences is not None:
            label += f" {confidences[i]:.2f}"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(out, (x1, y1 - th - 4), (x1 + tw, y1), color, -1)
        cv2.putText(out, label, (x1, y1 - 3),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return out


def _make_tracker_yaml(base_yaml_name, hp_overrides):
    """Write a temp tracker yaml with HP overrides; return its path."""
    import ultralytics
    shipped = Path(ultralytics.__file__).parent / "cfg" / "trackers" / base_yaml_name
    with open(shipped) as f:
        cfg = _yaml.safe_load(f)
    cfg.update(hp_overrides)
    tmp = Path(tempfile.mktemp(suffix=".yaml"))
    with open(tmp, "w") as f:
        _yaml.dump(cfg, f)
    return tmp


def run_tracker(model_path, video_paths, output_dir, tracker_yaml,
                hp_overrides=None, conf=CONF, iou=IOU, pp_overrides=None):
    """Run YOLO + tracker on a list of video files.
    Returns {video_stem: list-of-rows}  each row = [frame,id,x1,y1,w,h,cf,cid]

    pp_overrides: dict with optional keys:
        pp_cc_iou    - cross-class NMS IoU threshold (None = disabled)
        pp_merge_time - track merge time gap frames (None = disabled)
        pp_merge_dist - track merge spatial distance %
    """
    pp = pp_overrides or {}
    cc_iou      = pp.get("pp_cc_iou",       CROSS_CLASS_IOU_THRESH if CROSS_CLASS_NMS else None)
    merge_time  = pp.get("pp_merge_time",    MERGE_TIME_THRESHOLD   if AUTO_MERGE_TRACKS else None)
    merge_dist  = pp.get("pp_merge_dist",    MERGE_DISTANCE_THRESH)
    merge_interp= pp.get("pp_merge_interp",  True)

    # Extract detection thresholds from tracker hp_overrides (not passed to tracker yaml)
    hp = dict(hp_overrides or {})
    run_conf = hp.pop("det_conf", conf)
    run_iou  = hp.pop("det_iou",  iou)

    tmp_yaml = None
    if hp:
        tmp_yaml     = _make_tracker_yaml(tracker_yaml, hp)
        tracker_arg  = str(tmp_yaml)
    else:
        tracker_arg  = tracker_yaml

    model = YOLO(str(model_path))
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    all_preds = {}

    try:
        for vp in video_paths:
            vp = Path(vp)
            if not vp.exists():
                print(f"  [SKIP] {vp.name}")
                continue
            results = model.track(
                source=str(vp), conf=run_conf, iou=run_iou,
                tracker=tracker_arg, persist=False, verbose=False, stream=True,
            )
            rows = []
            for frame_idx, r in enumerate(results, 1):
                if r.boxes is None or r.boxes.id is None:
                    continue
                frame_rows = []
                for box in r.boxes:
                    if box.id is None:
                        continue
                    tid = int(box.id.item())
                    cid = int(box.cls.item())
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    frame_rows.append([frame_idx, tid, x1, y1, x2-x1, y2-y1,
                                       float(box.conf.item()), cid])
                # ── cross-class NMS ──────────────────────────────────────
                frame_rows = _cross_class_nms(frame_rows, cc_iou_thresh=cc_iou)
                rows.extend(frame_rows)
            # ── track merging ────────────────────────────────────────────
            rows = _merge_track_rows(rows, merge_time=merge_time, merge_dist=merge_dist,
                                        interpolate=merge_interp)
            pred_file = output_dir / f"{vp.stem}.txt"
            pred_file.write_text("\n".join(
                f"{r[0]},{r[1]},{r[2]:.2f},{r[3]:.2f},{r[4]:.2f},{r[5]:.2f},{r[6]:.4f},{r[7]}"
                for r in rows))
            all_preds[vp.stem] = rows
            # ── print GT vs pred track counts ────────────────────────────
            gt_file = MOT_DIR / "test" / vp.stem / "gt" / "gt.txt"
            if gt_file.exists():
                _gt = np.loadtxt(str(gt_file), delimiter=",")
                if _gt.ndim == 1: _gt = _gt.reshape(1, -1)
                if _gt.size > 0 and _gt.shape[1] >= 8:
                    gt_counts = {cls: 0 for cls in CLASSES}
                    for ci, cls in enumerate(CLASSES):
                        mask = _gt[:, 7].astype(int) == (ci + 1)
                        gt_counts[cls] = len(set(_gt[mask, 1].astype(int))) if mask.any() else 0
                    pred_counts = {cls: 0 for cls in CLASSES}
                    if rows:
                        _pr = np.array(rows)
                        for ci, cls in enumerate(CLASSES):
                            mask = _pr[:, 7].astype(int) == ci
                            pred_counts[cls] = len(set(_pr[mask, 1].astype(int))) if mask.any() else 0
                    gt_str   = "  ".join(f"{c}={gt_counts[c]}"   for c in CLASSES)
                    pred_str = "  ".join(f"{c}={pred_counts[c]}" for c in CLASSES)
                    print(f"  {vp.name}: {len(rows)} dets  |  GT [{gt_str}]  Pred [{pred_str}]")
                else:
                    print(f"  {vp.name}: {len(rows)} detections  (no GT annotations)")
            else:
                print(f"  {vp.name}: {len(rows)} detections  (no GT file)")
    finally:
        if tmp_yaml and tmp_yaml.exists():
            tmp_yaml.unlink()
    return all_preds


def compute_mot_metrics(gt_dir, pred_dir):
    """Compute MOTA / IDF1 / ID-Sw using motmetrics (symlink-aware)."""
    try:
        import motmetrics as mm
    except ImportError:
        print("  motmetrics not installed — run: pip install motmetrics")
        return {"MOTA": None, "IDF1": None, "ID_Sw": None}
    # motmetrics<=1.4.0 uses np.asfarray which was removed in NumPy 2.0
    if not hasattr(np, "asfarray"):
        np.asfarray = lambda a, dtype=float: np.asarray(a, dtype=dtype)

    accs, seq_names = [], []
    for gt_file in sorted(Path(gt_dir).glob("*/gt/gt.txt")):
        seq = gt_file.parent.parent.name
        pred_file = Path(pred_dir) / f"{seq}.txt"
        if not pred_file.exists():
            continue
        gt   = np.loadtxt(gt_file, delimiter=",")
        pred = (np.loadtxt(pred_file, delimiter=",")
                if pred_file.stat().st_size > 0 else np.zeros((0, 8)))
        if gt.ndim == 1:   gt   = gt.reshape(1, -1)
        if pred.ndim == 1: pred = pred.reshape(1, -1)
        # Skip sequences with no GT annotations (negative/unannotated chunks)
        if gt.shape[0] == 0 or gt.ndim < 2 or gt.shape[1] == 0:
            continue
        acc = mm.MOTAccumulator(auto_id=True)
        for fn in sorted(set(gt[:, 0].astype(int))):
            gt_fn = gt[gt[:, 0] == fn]
            pr_fn = pred[pred[:, 0] == fn] if len(pred) else np.zeros((0, 8))
            dist = (mm.distances.iou_matrix(gt_fn[:, 2:6],
                                            pr_fn[:, 2:6] if len(pr_fn) else np.zeros((0,4)),
                                            max_iou=0.5)
                    if len(gt_fn) and len(pr_fn)
                    else np.full((len(gt_fn), max(len(pr_fn),0)), np.nan))
            acc.update(gt_fn[:, 1].astype(int).tolist(),
                       pr_fn[:, 1].astype(int).tolist() if len(pr_fn) else [],
                       dist)
        accs.append(acc); seq_names.append(seq)

    if not accs:
        return {"MOTA": None, "IDF1": None, "ID_Sw": None}
    mh = mm.metrics.create()
    summary = mh.compute_many(accs, metrics=["mota","idf1","num_switches"],
                               names=seq_names, generate_overall=True)
    row = summary.loc["OVERALL"]
    return {"MOTA": float(row["mota"]), "IDF1": float(row["idf1"]),
            "ID_Sw": int(row["num_switches"])}


def tracker_name(tracker_yaml):
    """Return a short display name for a tracker yaml path."""
    return Path(tracker_yaml).stem


# ── Post-processing: cross-class NMS ──────────────────────────────────────
CROSS_CLASS_NMS        = True
CROSS_CLASS_IOU_THRESH = 0.5
AUTO_MERGE_TRACKS      = True
MERGE_TIME_THRESHOLD   = 30    # frames
MERGE_DISTANCE_THRESH  = 30.0  # % of frame dimensions

def _iou(b1, b2):
    """IoU for boxes in (x,y,w,h) format (pixel coords)."""
    ax1,ay1 = b1[0], b1[1]; ax2,ay2 = b1[0]+b1[2], b1[1]+b1[3]
    bx1,by1 = b2[0], b2[1]; bx2,by2 = b2[0]+b2[2], b2[1]+b2[3]
    ix1,iy1 = max(ax1,bx1), max(ay1,by1)
    ix2,iy2 = min(ax2,bx2), min(ay2,by2)
    inter = max(0,ix2-ix1)*max(0,iy2-iy1)
    union = b1[2]*b1[3] + b2[2]*b2[3] - inter
    return inter/union if union > 0 else 0.0

def _cx(box): return box[0]+box[2]/2
def _cy(box): return box[1]+box[3]/2

def _cross_class_nms(dets, cc_iou_thresh=None):
    """dets: list of [frame,tid,x,y,w,h,conf,cid]. Returns filtered list.
    cc_iou_thresh=None disables cross-class NMS entirely."""
    if cc_iou_thresh is None or len(dets) < 2:
        return dets
    dets = sorted(dets, key=lambda d: -d[6])   # sort by conf desc
    keep = [True]*len(dets)
    for i in range(len(dets)):
        if not keep[i]: continue
        for j in range(i+1, len(dets)):
            if not keep[j]: continue
            if dets[i][7] == dets[j][7]: continue   # same class — skip
            if _iou(dets[i][2:6], dets[j][2:6]) >= cc_iou_thresh:
                keep[j] = False
    return [d for d, k in zip(dets, keep) if k]

def _merge_track_rows(rows, merge_time=None, merge_dist=30.0, interpolate=True):
    """rows: list of [frame,tid,x,y,w,h,conf,cid]. Returns merged rows.
    merge_time=None disables merging entirely.
    interpolate=True fills gap frames between merged segments with linearly
    interpolated boxes (conf=0.0, marked as synthetic).
    """
    if merge_time is None:
        return rows
    # Build per-track info
    from collections import defaultdict
    tracks = defaultdict(list)
    for r in rows:
        tracks[r[1]].append(r)
    # Sort each track by frame
    for tid in tracks:
        tracks[tid].sort(key=lambda r: r[0])
    # Build metadata list
    meta = [{"tid": tid,
             "cid": segs[0][7],
             "start": segs[0][0],
             "end":   segs[-1][0],
             "rows":  segs,
             "merged_into": None}
            for tid, segs in tracks.items()]
    meta.sort(key=lambda m: m["start"])
    merged = 0
    for j in range(len(meta)):
        if meta[j]["merged_into"] is not None: continue
        candidates = []
        for i in range(j):
            if meta[i]["merged_into"] is not None: continue
            if meta[i]["cid"] != meta[j]["cid"]: continue
            gap = meta[j]["start"] - meta[i]["end"]
            if gap < 0 or gap > merge_time: continue
            # spatial distance (pixels)
            last  = meta[i]["rows"][-1]; first = meta[j]["rows"][0]
            dist = ((_cx(last[2:6])-_cx(first[2:6]))**2 + (_cy(last[2:6])-_cy(first[2:6]))**2)**0.5
            max_dim = max(last[4], last[5], first[4], first[5], 1)
            dist_pct = dist / max_dim * 100
            if dist_pct <= merge_dist:
                candidates.append((i, dist_pct, gap))
        if candidates:
            candidates.sort(key=lambda c: (c[1], c[2]))
            best_i = candidates[0][0]

            # ── Linear interpolation across the gap ───────────────────────
            gap_rows = []
            if interpolate:
                r_end   = meta[best_i]["rows"][-1]   # last row of earlier track
                r_start = meta[j]["rows"][0]          # first row of later track
                f0, f1  = int(r_end[0]), int(r_start[0])
                if f1 - f0 > 1:
                    for f in range(f0 + 1, f1):
                        t = (f - f0) / (f1 - f0)     # 0 < t < 1
                        ix = r_end[2] + t * (r_start[2] - r_end[2])
                        iy = r_end[3] + t * (r_start[3] - r_end[3])
                        iw = r_end[4] + t * (r_start[4] - r_end[4])
                        ih = r_end[5] + t * (r_start[5] - r_end[5])
                        gap_rows.append([f, meta[j]["tid"], ix, iy, iw, ih,
                                         0.0, meta[j]["cid"]])

            meta[j]["rows"] = meta[best_i]["rows"] + gap_rows + meta[j]["rows"]
            meta[j]["start"] = meta[best_i]["start"]
            meta[best_i]["merged_into"] = meta[j]["tid"]
            merged += 1
    if merged:
        n_interp = sum(1 for m in meta if m["merged_into"] is None
                       for r in m["rows"] if r[6] == 0.0)
        print(f"    [merge] merged {merged} track(s), interpolated {n_interp} gap frame(s)")
    out = []
    for m in meta:
        if m["merged_into"] is None:
            out.extend(m["rows"])
    return out

print("Helpers ready.")


## 2 — Hyperparameter Search (test set, both trackers)

For each tracker we sample up to `HP_MAX_TRIALS` random combinations from
`HP_GRID`, run inference on test-set videos using the **final** YOLO model,
and select the configuration with the highest MOTA.

> Lower `HP_MAX_TRIALS` in Section 0 to reduce runtime.

In [ ]:
if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Final weights not found: {FINAL_WEIGHTS}\n"
        "Run 02_train_yolo.ipynb — Section 2 (Final Model) first.")

test_chunk_names = splits["mot_test_chunks"]
all_test_videos = sorted([
    CHUNKS_DIR / fname
    for fname in test_chunk_names
    if (CHUNKS_DIR / fname).exists()
])
print(f"Total test videos: {len(all_test_videos)}")

# ── HP-search subset ──────────────────────────────────────────────────────
random.seed(42)
n_subset = max(1, int(len(all_test_videos) * HP_SEARCH_SUBSET))
search_videos = random.sample(all_test_videos, n_subset)
print(f"HP search subset : {len(search_videos)} / {len(all_test_videos)} "
      f"({HP_SEARCH_SUBSET*100:.0f}%)")

# ── Separate tracker and post-processing keys ─────────────────────────────
_TRACKER_KEYS = ["det_conf", "det_iou",
                 "track_high_thresh", "track_low_thresh", "new_track_thresh",
                 "track_buffer", "match_thresh"]
_PP_KEYS      = ["pp_cc_iou", "pp_merge_time", "pp_merge_dist", "pp_merge_interp"]

tracker_grid = {k: HP_GRID[k] for k in _TRACKER_KEYS if k in HP_GRID}
pp_grid      = {k: HP_GRID[k] for k in _PP_KEYS      if k in HP_GRID}

tracker_combos = list(itertools.product(*tracker_grid.values()))
pp_combos      = list(itertools.product(*pp_grid.values()))
all_combos     = [dict(**dict(zip(tracker_grid.keys(), tc)),
                       **dict(zip(pp_grid.keys(),      pc)))
                  for tc, pc in itertools.product(tracker_combos, pp_combos)]

random.shuffle(all_combos)
combos = all_combos[:HP_MAX_TRIALS]
print(f"HP trials per tracker: {len(combos)} / {len(all_combos)} total")

best_configs  = {}   # tracker yaml -> best hp dict
best_metrics  = {}   # tracker yaml -> best metrics dict
hp_search_log = []

# Determine whether higher or lower is better for the chosen metric
_METRIC_MAXIMIZE = {"MOTA": True, "IDF1": True, "ID_Sw": False, "ACE": False}
_metric_maximize = _METRIC_MAXIMIZE.get(HP_SEARCH_METRIC, True)
_best_init       = -float("inf") if _metric_maximize else float("inf")

def _is_better(new_val, best_val):
    if new_val is None: return False
    return new_val > best_val if _metric_maximize else new_val < best_val

for tracker in TRACKERS:
    tname = tracker_name(tracker)
    print(f"\n{'='*60}\nHP search — {tname}  ({len(search_videos)} videos)  [optimising {HP_SEARCH_METRIC}]\n{'='*60}")
    best_score = _best_init

    for trial_idx, combo in enumerate(combos):
        tracker_hp = {k: v for k, v in combo.items() if k in _TRACKER_KEYS}
        pp_hp      = {k: v for k, v in combo.items() if k in _PP_KEYS}
        det_hp     = {k: tracker_hp[k] for k in ("det_conf", "det_iou") if k in tracker_hp}

        print(f"\n  Trial {trial_idx+1}/{len(combos)}")
        print(f"    detection: {det_hp}")
        print(f"    tracker  : { {k:v for k,v in tracker_hp.items() if k not in det_hp} }")
        print(f"    post-proc: {pp_hp}")

        pred_dir = TRACKER_RUNS / tname / "hp_search" / f"trial_{trial_idx:03d}"
        run_tracker(FINAL_WEIGHTS, search_videos, pred_dir, tracker,
                    hp_overrides=tracker_hp, pp_overrides=pp_hp)
        m = compute_mot_metrics(MOT_DIR / "test", pred_dir)

        # Compute ACE for count-based optimisation
        if HP_SEARCH_METRIC == "ACE":
            df_cnt = compute_count_metrics(MOT_DIR / "test", pred_dir, CLASSES)
            macro_row = df_cnt[df_cnt["class"] == "MACRO_AVG"]
            ace_val = float(macro_row["ACE"].values[0]) if not macro_row.empty else None
            m["ACE"] = ace_val
        else:
            m["ACE"] = None

        score = m.get(HP_SEARCH_METRIC)
        print(f"    → MOTA={m['MOTA']}  IDF1={m['IDF1']}  ID_Sw={m['ID_Sw']}  ACE={m['ACE']}  [{HP_SEARCH_METRIC}={score}]")
        hp_search_log.append({"tracker": tname, "trial": trial_idx, **combo, **m})
        if _is_better(score, best_score):
            best_score = score
            best_configs[tracker]  = combo
            best_metrics[tracker]  = m

    print(f"\n  ✓ Best {HP_SEARCH_METRIC}={best_score}")
    print(f"    tracker  : { {k:v for k,v in best_configs.get(tracker,{}).items() if k in _TRACKER_KEYS} }")
    print(f"    post-proc: { {k:v for k,v in best_configs.get(tracker,{}).items() if k in _PP_KEYS} }")

df_hp = pd.DataFrame(hp_search_log)
df_hp.to_csv(TRACKER_RUNS / "hp_search_log.csv", index=False)
print("\nBest configs per tracker:")
for t, hp in best_configs.items():
    print(f"  {tracker_name(t):20s}: {hp}")


## 3 — Final Evaluation with Best Hyperparameters

In [ ]:
final_results = {}

# ── Use the same subset as HP search for fast turnaround.
# ── Swap search_videos → all_test_videos below for a full evaluation.
eval_videos = search_videos

for tracker in TRACKERS:
    tname = tracker_name(tracker)
    best_hp  = best_configs.get(tracker, {})
    tracker_hp = {k: v for k, v in best_hp.items() if k in _TRACKER_KEYS}
    pp_hp      = {k: v for k, v in best_hp.items() if k in _PP_KEYS}
    pred_dir = TRACKER_RUNS / tname / "final_eval"
    print(f"Running {tname} (best HP, {len(eval_videos)}/{len(all_test_videos)} videos)...")
    print(f"  tracker  : {tracker_hp}")
    print(f"  post-proc: {pp_hp}")
    run_tracker(FINAL_WEIGHTS, eval_videos, pred_dir, tracker,
                hp_overrides=tracker_hp, pp_overrides=pp_hp)
    m = compute_mot_metrics(MOT_DIR / "test", pred_dir)
    final_results[tname] = {"metrics": m, "hp": best_hp, "pred_dir": str(pred_dir)}
    print(f"  → MOTA={m['MOTA']}  IDF1={m['IDF1']}  ID_Sw={m['ID_Sw']}")

df_final = pd.DataFrame([{"tracker": k, **v["metrics"]} for k, v in final_results.items()])
print("\n── Final test-set results ──")
print(df_final.round(4).to_string(index=False))
df_final.to_csv(TRACKER_RUNS / "test_metrics.csv", index=False)
with open(TRACKER_RUNS / "test_metrics.json", "w") as f:
    json.dump({k: v["metrics"] for k, v in final_results.items()}, f, indent=2)
with open(TRACKER_RUNS / "best_configs.json", "w") as f:
    json.dump({k: v["hp"] for k, v in final_results.items()}, f, indent=2)


## 3b — Per-Class Instance Count Metrics

For each test video, counts unique track IDs per class in predictions vs ground truth,
then averages across videos.

Metrics per class:
- **GT** / **Pred**: mean track count across videos
- **CE** (Count Error): mean(pred − gt) — signed, shows systematic over/under-detection
- **ACE** (Abs Count Error): mean |pred − gt|
- **CA** (Count Accuracy): mean min(pred,gt)/max(pred,gt) ∈ [0,1]

In [ ]:
def _count_tracks_per_class(txt_file, n_classes, col_cid=7, col_tid=1):
    """Return array of unique track counts per class from a MOT-format .txt file."""
    counts = np.zeros(n_classes, dtype=int)
    if not Path(txt_file).exists() or Path(txt_file).stat().st_size == 0:
        return counts
    data = np.loadtxt(txt_file, delimiter=",")
    if data.ndim == 1: data = data.reshape(1, -1)
    if data.shape[0] == 0: return counts
    for cid in range(n_classes):
        mask = data[:, col_cid].astype(int) == cid
        counts[cid] = len(set(data[mask, col_tid].astype(int))) if mask.any() else 0
    return counts


def compute_count_metrics(gt_dir, pred_dir, classes):
    """Compute per-class count metrics averaged over all test sequences.

    Returns a DataFrame with columns:
        class, GT_mean, Pred_mean, CE, ACE, CA

    Note: GT files store class IDs 1-based (1=classes[0], 2=classes[1], ...),
    while pred files store 0-based class IDs matching CLASSES order.
    """
    import os as _os
    n = len(classes)
    gt_counts_all  = []   # list of arrays, one per sequence
    pred_counts_all = []

    gt_dir = Path(gt_dir)
    for seq in sorted(_os.listdir(gt_dir)):
        gt_file  = gt_dir / seq / "gt" / "gt.txt"
        pred_file = Path(pred_dir) / f"{seq}.txt"
        if not gt_file.exists(): continue

        gt_data = np.loadtxt(gt_file, delimiter=",")
        if gt_data.ndim == 1: gt_data = gt_data.reshape(1, -1)
        if gt_data.shape[0] == 0 or gt_data.shape[1] < 8:
            continue  # skip unannotated sequences

        gt_c = np.zeros(n, dtype=int)
        for i in range(n):
            gt_cid = i + 1   # GT uses 1-based class IDs
            mask = gt_data[:, 7].astype(int) == gt_cid
            gt_c[i] = len(set(gt_data[mask, 1].astype(int))) if mask.any() else 0

        pred_c = _count_tracks_per_class(pred_file, n)   # pred is 0-based

        gt_counts_all.append(gt_c)
        pred_counts_all.append(pred_c)

    if not gt_counts_all:
        return pd.DataFrame()

    gt_arr   = np.array(gt_counts_all,   dtype=float)   # (n_seqs, n_classes)
    pred_arr = np.array(pred_counts_all, dtype=float)

    gt_mean   = gt_arr.mean(axis=0)
    pred_mean = pred_arr.mean(axis=0)
    ce        = (pred_arr - gt_arr).mean(axis=0)
    ace       = np.abs(pred_arr - gt_arr).mean(axis=0)

    # Count Accuracy: per video, per class = min(p,g)/max(p,g), then average
    denom = np.maximum(pred_arr, gt_arr)
    numer = np.minimum(pred_arr, gt_arr)
    ca_per = np.where(denom > 0, numer / denom, 1.0)   # both=0 → perfect
    ca     = ca_per.mean(axis=0)

    rows = []
    for i, cls in enumerate(classes):
        rows.append({"class": cls,
                     "GT_mean": round(gt_mean[i], 2),
                     "Pred_mean": round(pred_mean[i], 2),
                     "CE": round(ce[i], 3),
                     "ACE": round(ace[i], 3),
                     "CA": round(ca[i], 3)})
    # macro average
    rows.append({"class": "MACRO_AVG",
                 "GT_mean": round(gt_mean.mean(), 2),
                 "Pred_mean": round(pred_mean.mean(), 2),
                 "CE": round(ce.mean(), 3),
                 "ACE": round(ace.mean(), 3),
                 "CA": round(ca.mean(), 3)})
    return pd.DataFrame(rows)


# ── Run for each tracker ───────────────────────────────────────────────────
count_results = {}
for tracker in TRACKERS:
    tname    = tracker_name(tracker)
    pred_dir = Path(final_results[tname]["pred_dir"])
    df_cnt   = compute_count_metrics(MOT_DIR / "test", pred_dir, CLASSES)
    count_results[tname] = df_cnt
    print(f"\n── {tname} — Per-Class Count Metrics ──")
    print(df_cnt.to_string(index=False))

# ── Visualize: grouped bar chart (ACE per class, one group per tracker) ───
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Per-Class Count Metrics — Test Set", fontweight="bold")
bar_colors = ["#4C72B0", "#DD8452"]

for ax, metric, ylabel in zip(axes,
                               ["CE",   "ACE",              "CA"],
                               ["Count Error (CE)",
                                "Abs Count Error (ACE)",
                                "Count Accuracy (CA)"]):
    class_labels = CLASSES
    x = np.arange(len(class_labels))
    width = 0.8 / len(TRACKERS)
    for ti, tracker in enumerate(TRACKERS):
        tname = tracker_name(tracker)
        df    = count_results[tname]
        vals  = [df.loc[df["class"] == c, metric].values[0]
                 if c in df["class"].values else 0
                 for c in class_labels]
        offset = (ti - len(TRACKERS)/2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=tname,
                      color=bar_colors[ti % len(bar_colors)], alpha=0.85)
    if metric == "CE":
        ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_xticks(x); ax.set_xticklabels(class_labels, rotation=30, ha="right")
    ax.set_ylabel(ylabel); ax.set_title(ylabel)
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out_path = TRACKER_RUNS / "count_metrics.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


## 4 — Metric Visualizations

In [ ]:
import seaborn as sns

tracker_names = [tracker_name(t) for t in TRACKERS]

# ── Bar chart ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = ["#4C72B0", "#DD8452"]
for ax, metric in zip(axes, ["MOTA", "IDF1", "ID_Sw"]):
    vals = [final_results[n]["metrics"].get(metric) or 0 for n in tracker_names]
    bars = ax.bar(tracker_names, vals, color=colors[:len(tracker_names)])
    ax.set_title(metric)
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}" if metric != "ID_Sw" else str(int(val)),
                ha="center", va="bottom", fontsize=10)
plt.suptitle("Tracker Comparison — Test Set (best HP)", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(TRACKER_RUNS / "metric_barplot.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {TRACKER_RUNS}/metric_barplot.png")

# ── HP search heat-map (MOTA vs two key params) ────────────────────────────
if "track_high_thresh" in HP_GRID and "match_thresh" in HP_GRID and not df_hp.empty:
    fig, axes = plt.subplots(1, len(TRACKERS), figsize=(7*len(TRACKERS), 5))
    if len(TRACKERS) == 1:
        axes = [axes]
    for ax, tracker in zip(axes, TRACKERS):
        name = tracker_name(tracker)
        sub = df_hp[df_hp["tracker"] == name].copy()
        if sub.empty or sub["MOTA"].isna().all():
            ax.set_title(f"{name} — no data"); continue
        pivot = sub.pivot_table(index="track_high_thresh", columns="match_thresh",
                                values="MOTA", aggfunc="mean")
        sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax,
                    linewidths=0.5, cbar_kws={"label": "MOTA"})
        ax.set_title(f"{name} — MOTA (track_high_thresh × match_thresh)")
    plt.tight_layout()
    plt.savefig(TRACKER_RUNS / "hp_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {TRACKER_RUNS}/hp_heatmap.png")


## 5 — Annotated Frame Snapshots

Extract 12 evenly-spaced frames from the first test video for each tracker
and display them in a grid.

In [ ]:
import math

# Pick demo from the eval subset so pred files are guaranteed to exist
demo_video = eval_videos[0] if eval_videos else None

def annotated_grid(video_path, model_path, tracker_yaml, hp={}, conf=CONF, iou=IOU, n_frames=12):
    tmp_yaml = _make_tracker_yaml(tracker_yaml, hp) if hp else None
    tracker_arg = str(tmp_yaml) if tmp_yaml else tracker_yaml
    model = YOLO(str(model_path))
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    results_list = list(model.track(
        source=str(video_path), conf=conf, iou=iou,
        tracker=tracker_arg, persist=False, verbose=False, stream=True))
    if tmp_yaml and tmp_yaml.exists():
        tmp_yaml.unlink()
    indices = [int(i*(total-1)/(n_frames-1)) for i in range(n_frames)]
    out_frames = []
    cap = cv2.VideoCapture(str(video_path))
    for fi in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret:
            continue
        r = results_list[min(fi, len(results_list)-1)]
        boxes, tids, cids, confs = [], [], [], []
        if r.boxes is not None and r.boxes.id is not None:
            for box in r.boxes:
                if box.id is None: continue
                boxes.append(box.xyxy[0].tolist())
                tids.append(int(box.id.item()))
                cids.append(int(box.cls.item()))
                confs.append(float(box.conf.item()))
        out_frames.append(draw_tracks(frame, boxes, tids, cids, CLASSES, confs))
    cap.release()
    return out_frames

if demo_video:
    for tracker in TRACKERS:
        tname = tracker_name(tracker)
        best_hp = best_configs.get(tracker, {})
        tracker_hp = {k: v for k, v in best_hp.items() if k in _TRACKER_KEYS}
        best_conf  = tracker_hp.pop("det_conf", CONF)
        best_iou   = tracker_hp.pop("det_iou",  IOU)
        frames = annotated_grid(demo_video, FINAL_WEIGHTS, tracker,
                                hp=tracker_hp, conf=best_conf, iou=best_iou, n_frames=12)
        cols = 4
        rows_n = math.ceil(len(frames) / cols)
        fig, axes = plt.subplots(rows_n, cols, figsize=(cols*4, rows_n*3))
        axes = axes.flatten()
        for ax, fr in zip(axes, frames):
            ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.axis("off")
        for ax in axes[len(frames):]:
            ax.axis("off")
        plt.suptitle(f"{tname} — annotated frames  ({demo_video.name})", fontsize=12)
        plt.tight_layout()
        out_path = TRACKER_RUNS / f"annotated_{tname}.png"
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Saved: {out_path}")
else:
    print("No test video found — skipping frame snapshots.")


## 6 — Trajectory Plot

Centroid path of every tracked object over the first test video for the
best-MOTA tracker, overlaid on the first frame.

In [ ]:
import matplotlib.cm as cm

best_tracker_name = max(
    final_results,
    key=lambda k: final_results[k]["metrics"].get("MOTA") or -1)
best_tracker_yaml = best_tracker_name + ".yaml"
best_hp = best_configs.get(best_tracker_yaml, {})

def _load_gt_trajectories(demo_video, mot_dir):
    """Load ground-truth trajectories for demo_video from the MOT test set.

    Returns dict: tid -> list of (cx, cy, frame, cid)  or empty dict if not found.
    """
    seq_name = Path(demo_video).stem
    gt_file  = Path(mot_dir) / "test" / seq_name / "gt" / "gt.txt"
    if not gt_file.exists():
        print(f"  [GT] gt.txt not found for {seq_name} — skipping GT overlay")
        return {}
    gt = np.loadtxt(gt_file, delimiter=",")
    if gt.ndim == 1: gt = gt.reshape(1, -1)
    trajs = {}
    for row in gt:
        fn, tid, x, y, w, h = int(row[0]), int(row[1]), row[2], row[3], row[4], row[5]
        cid = int(row[7]) if gt.shape[1] > 7 else 0
        trajs.setdefault(tid, []).append((x + w/2, y + h/2, fn, cid))
    for tid in trajs:
        trajs[tid].sort(key=lambda p: p[2])
    return trajs


def _draw_trajectory_ax(ax, trajectories, title, bg):
    ax.imshow(cv2.cvtColor(bg, cv2.COLOR_BGR2RGB), alpha=0.35)
    for tid, pts in trajectories.items():
        cid   = pts[0][3]
        color = [c/255 for c in class_color(cid)]
        xs, ys = [p[0] for p in pts], [p[1] for p in pts]
        ax.plot(xs, ys, color=color, linewidth=1.5, alpha=0.8)
        ax.scatter(xs[-1], ys[-1], color=color, s=25, zorder=5)
        ax.text(xs[-1]+3, ys[-1]-3, f"#{tid}", fontsize=6, color=color)
    ax.legend(handles=[
        mpatches.Patch(color=[c/255 for c in class_color(i)], label=cls)
        for i, cls in enumerate(CLASSES)
    ], loc="upper right", fontsize=7, framealpha=0.7, ncol=2)
    ax.set_title(title, fontsize=11)
    ax.axis("off")


if demo_video:
    print(f"Trajectory plot — {best_tracker_name}  ({demo_video.name})")

    # ── Predicted trajectories ────────────────────────────────────────────
    _best_hp_traj = best_configs.get(best_tracker_yaml, {})
    _traj_tracker_hp = {k: v for k, v in _best_hp_traj.items() if k in _TRACKER_KEYS}
    _traj_conf = _traj_tracker_hp.pop("det_conf", CONF)
    _traj_iou  = _traj_tracker_hp.pop("det_iou",  IOU)
    tmp_yaml    = _make_tracker_yaml(best_tracker_yaml, _traj_tracker_hp) if _traj_tracker_hp else None
    tracker_arg = str(tmp_yaml) if tmp_yaml else best_tracker_yaml
    model = YOLO(str(FINAL_WEIGHTS))
    pred_trajectories = {}
    for frame_idx, r in enumerate(
        model.track(source=str(demo_video), conf=_traj_conf, iou=_traj_iou,
                    tracker=tracker_arg, persist=False, verbose=False, stream=True), 1):
        if r.boxes is None or r.boxes.id is None: continue
        for box in r.boxes:
            if box.id is None: continue
            tid = int(box.id.item())
            cid = int(box.cls.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            pred_trajectories.setdefault(tid, []).append(((x1+x2)/2, (y1+y2)/2, frame_idx, cid))
    if tmp_yaml and tmp_yaml.exists(): tmp_yaml.unlink()

    # ── Ground-truth trajectories ─────────────────────────────────────────
    gt_trajectories = _load_gt_trajectories(demo_video, MOT_DIR)

    # ── Background frame ─────────────────────────────────────────────────
    cap = cv2.VideoCapture(str(demo_video))
    ret, bg = cap.read(); cap.release()
    if not ret: bg = np.zeros((720, 1280, 3), dtype=np.uint8)

    # ── Side-by-side plot ─────────────────────────────────────────────────
    n_cols = 2 if gt_trajectories else 1
    fig, axes = plt.subplots(1, n_cols, figsize=(12 * n_cols, 7))
    if n_cols == 1: axes = [axes]

    _draw_trajectory_ax(axes[0], pred_trajectories,
                        f"Predicted — {best_tracker_name}  ({demo_video.name})", bg)
    if gt_trajectories:
        _draw_trajectory_ax(axes[1], gt_trajectories,
                            f"Ground Truth  ({demo_video.name})", bg)

    plt.tight_layout()
    out_path = TRACKER_RUNS / "trajectories.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")
else:
    print("No test video found — skipping trajectory plot.")


## 7 — Real-Time Interactive Tracking

Opens an OpenCV window and runs the best tracker live on a chosen video.
Press **`q`** to quit, **`space`** to pause / resume.

Change `RT_VIDEO` to any `.mp4` path, or `0` for webcam.

In [ ]:
# ── Configure ─────────────────────────────────────────────────────────────
RT_VIDEO   = demo_video          # or Path("path/to/video.mp4") or 0 for webcam
RT_TRACKER = best_tracker_yaml
RT_HP      = best_configs.get(RT_TRACKER, {})
# ──────────────────────────────────────────────────────────────────────────

if RT_VIDEO is None and not isinstance(RT_VIDEO, int):
    print("RT_VIDEO is None — skipping real-time cell.")
else:
    _tmp_yaml = _make_tracker_yaml(RT_TRACKER, RT_HP) if RT_HP else None
    _t_arg    = str(_tmp_yaml) if _tmp_yaml else RT_TRACKER
    _model    = YOLO(str(FINAL_WEIGHTS))
    _src      = str(RT_VIDEO) if not isinstance(RT_VIDEO, int) else RT_VIDEO

    _cap = cv2.VideoCapture(_src)
    if not _cap.isOpened():
        print(f"Cannot open: {RT_VIDEO}")
    else:
        print("Press 'q' to quit, 'space' to pause/resume.")
        _paused = False
        for _r in _model.track(source=_src, conf=CONF, iou=IOU,
                                tracker=_t_arg, persist=True, verbose=False, stream=True):
            _ret, _frame = _cap.read()
            if not _ret: break
            _boxes, _tids, _cids, _confs = [], [], [], []
            if _r.boxes is not None and _r.boxes.id is not None:
                for _b in _r.boxes:
                    if _b.id is None: continue
                    _boxes.append(_b.xyxy[0].tolist())
                    _tids.append(int(_b.id.item()))
                    _cids.append(int(_b.cls.item()))
                    _confs.append(float(_b.conf.item()))
            _vis = draw_tracks(_frame, _boxes, _tids, _cids, CLASSES, _confs)
            cv2.imshow("KARTector — real-time  [q=quit  space=pause]", _vis)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"): break
            if key == ord(" "):
                _paused = not _paused
                while _paused:
                    k2 = cv2.waitKey(50) & 0xFF
                    if k2 in (ord(" "), ord("q")): _paused = False
        _cap.release()
        cv2.destroyAllWindows()
        if _tmp_yaml and _tmp_yaml.exists(): _tmp_yaml.unlink()
        print("Real-time session ended.")
